# 00 — NumPy Basics (ARIS Phase 0)

**Purpose:** prove NumPy fluency before Phase 2 lap-time predictor work.

**Self-test target (from `SKILLS-MASTERY.md` §2.1):** broadcasting, vectorisation, slicing-as-view, `np.where`, `np.linalg`, random-with-seed.

Run top-to-bottom on a fresh kernel before commit. Clear all outputs, then commit.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)  # reproducible random; never use np.random.seed in new code
print(np.__version__)

## 1. Array creation

Five ways you'll actually use:
- `np.array([...])` — from a Python list
- `np.zeros(shape)`, `np.ones(shape)` — pre-allocate
- `np.arange(start, stop, step)` — like `range` but returns an ndarray
- `np.linspace(start, stop, n)` — n evenly-spaced points (inclusive of both ends)
- `rng.normal(loc, scale, size)` — random samples

In [ ]:
a = np.array([1, 2, 3, 4, 5])
z = np.zeros((2, 3))
o = np.ones(4)
rng_arr = np.arange(0, 10, 2)
lin = np.linspace(0, 1, 5)
noise = rng.normal(loc=0.0, scale=0.1, size=5)

print("a:", a)
print("zeros:\n", z)
print("ones:", o)
print("arange:", rng_arr)
print("linspace:", lin)
print("noise:", noise)

## 2. Shape, dtype, ndim

Three attributes you check when something doesn't work:
- `arr.shape` — tuple of dimensions
- `arr.dtype` — element type (`float64`, `int32`, `bool`)
- `arr.ndim` — number of dimensions

In [ ]:
m = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("shape:", m.shape)
print("dtype:", m.dtype)
print("ndim:", m.ndim)
print("size (total elements):", m.size)

## 3. Indexing & slicing

Same syntax as Python lists, generalised to multiple dims. `a[i, j]` not `a[i][j]`.

**Important: slicing returns a *view*, not a copy.** Modifying the slice modifies the original. Use `.copy()` when you need independence.

In [ ]:
x = np.arange(12).reshape(3, 4)
print("x:\n", x)
print("x[1, 2]:", x[1, 2])           # single element
print("x[0]:", x[0])                  # first row
print("x[:, 1]:", x[:, 1])            # second column
print("x[1:, :2]:\n", x[1:, :2])      # rows 1+, cols 0..1
print("x[::-1]:\n", x[::-1])          # reverse rows

In [ ]:
# View vs copy — this is the most common NumPy bug
original = np.arange(5)
view = original[1:4]
view[0] = 999
print("original after writing to view:", original)  # original is mutated!

original2 = np.arange(5)
copy = original2[1:4].copy()
copy[0] = 999
print("original2 after writing to copy:", original2)  # untouched

## 4. Boolean indexing & np.where

Filter arrays without writing a loop.
- `arr[mask]` — keep elements where mask is True
- `np.where(cond, a, b)` — element-wise ternary

In [ ]:
lap_times = np.array([91.2, 92.4, 91.8, 95.6, 91.5, 92.0, 99.1])

fast = lap_times[lap_times < 92.0]
print("fast laps (<92s):", fast)

# np.where — flag laps as 'clean' or 'traffic'
labels = np.where(lap_times < 93.0, "clean", "traffic")
print("labels:", labels)

## 5. Broadcasting

Operations between arrays of different shapes — NumPy stretches the smaller one. Rule: align trailing dims; size-1 or missing dims expand.

Examples:
- `(3, 4) + (4,)` → `(3, 4)` (the `(4,)` is added to each row)
- `(3, 1) + (1, 4)` → `(3, 4)` (outer-product-like expansion)
- `(3, 4) + (3,)` → **ERROR** (trailing dims don't align)

In [ ]:
# Three drivers, four laps — fuel correction per lap (lighter car each lap)
raw_times = np.array([
    [92.5, 92.3, 92.1, 91.9],   # driver 0
    [93.1, 92.9, 92.7, 92.5],   # driver 1
    [91.8, 91.6, 91.4, 91.2],   # driver 2
])  # shape (3, 4)

fuel_correction = np.array([0.0, -0.05, -0.10, -0.15])  # shape (4,) — per-lap fuel-burn benefit

corrected = raw_times + fuel_correction  # broadcasts (4,) across each row of (3,4)
print("corrected:\n", corrected)

## 6. Vectorised math

Apply maths element-wise without a Python loop. ~100× faster on large arrays.

In [ ]:
speeds_kmh = np.array([280.0, 295.0, 310.0, 320.0, 305.0])

speeds_ms = speeds_kmh / 3.6                 # element-wise divide
kinetic = 0.5 * 798.0 * speeds_ms ** 2       # KE = 1/2 m v^2 (798 kg ≈ F1 car + driver)
log_speeds = np.log(speeds_ms)

print("speeds (m/s):", speeds_ms)
print("kinetic energy (J):", kinetic)
print("log speeds:", log_speeds)

## 7. Aggregations

`mean, std, min, max, sum, median` etc. all take an `axis` argument.

In [ ]:
# Same lap-time matrix from §5 — drivers x laps
print("shape:", raw_times.shape)
print("overall mean:", raw_times.mean())
print("per-driver mean (axis=1):", raw_times.mean(axis=1))   # collapse columns -> (3,)
print("per-lap mean    (axis=0):", raw_times.mean(axis=0))   # collapse rows    -> (4,)
print("argmin per driver:", raw_times.argmin(axis=1))         # which lap was each driver's fastest

## 8. Random — seeded for reproducibility

Always use `np.random.default_rng(seed)`. The legacy global `np.random.seed()` is deprecated for new code.

In [ ]:
rng2 = np.random.default_rng(seed=0)
samples = rng2.normal(loc=92.0, scale=0.4, size=1000)   # 1000 simulated lap times
print(f"simulated mean: {samples.mean():.3f}")
print(f"simulated std:  {samples.std():.3f}")
print(f"% sub-91.5:     {(samples < 91.5).mean() * 100:.1f}%")

## 9. np.linalg — solve a linear system

OLS regression closed-form: `β = (XᵀX)⁻¹Xᵀy`. Below: fit a linear lap-time model from two features (lap-in-stint, fuel-mass) without using sklearn.

In [ ]:
# Synthetic stint: 20 laps, fuel decreasing 1.5 kg/lap, lap-time ~ const + 0.04*lap + 0.03*fuel + noise
n = 20
lap = np.arange(n)
fuel = 100.0 - 1.5 * lap
true_beta = np.array([90.0, 0.04, 0.03])  # intercept, lap, fuel
X = np.column_stack([np.ones(n), lap, fuel])           # design matrix shape (20, 3)
y = X @ true_beta + rng.normal(0, 0.1, size=n)         # @ is matrix multiply

# Closed-form OLS
beta_hat = np.linalg.solve(X.T @ X, X.T @ y)
print("true beta:    ", true_beta)
print("recovered beta:", beta_hat.round(3))

## 10. ARIS-flavoured exercise — fuel-corrected pace ranking

Given a `(drivers, laps)` lap-time matrix and a per-lap fuel-correction vector, produce a sorted ranking of drivers by their **fuel-corrected mean pace** in one line of NumPy (no loops, no pandas).

In [ ]:
drivers = np.array(["VER", "HAM", "LEC", "NOR"])
lap_times_grid = np.array([
    [92.4, 92.2, 92.0, 91.8],   # VER
    [92.8, 92.6, 92.4, 92.3],   # HAM
    [92.5, 92.3, 92.0, 91.9],   # LEC
    [92.7, 92.5, 92.3, 92.1],   # NOR
])
fuel_correction = np.array([0.0, -0.05, -0.10, -0.15])

# fuel-corrected mean pace, sorted ascending (fastest first)
ranking = drivers[np.argsort((lap_times_grid + fuel_correction).mean(axis=1))]
print("fastest -> slowest by fuel-corrected pace:", ranking)

## Takeaway

If §1–§9 ran clean and the §10 one-liner felt natural, the Phase 0 NumPy bar is met:
*"can do simple cases, struggles with edge cases"* (rubric score 2).

Edge cases that are **not** covered here and will bite later:
- `np.einsum` (worth knowing but rarely needed)
- Memory layout (`C` vs `F` order) — only matters at scale
- `np.ma` (masked arrays) — pandas covers most missing-data needs

Next: pandas (notebook `01` or `02`).